# Example code to get alternative pairs using the feature-space pairing algorithm

In [ ]:
import geopandas as gpd
import pandas as pd
import xarray as xr
import numpy as np
import tqdm as tqdm

import sys
sys.path.append('..')

from chap1_modules.gedi_pairs import pairing_algorithms

*A previously downloaded zarr of all undisturbed GEDI data in Spain id needed to continue witht this example.*

**Please contact the author to get that file**

In [19]:
ds = xr.open_dataset('data/Spain/GEDI_strata_filtered_disturbance.zarr')

undisturbed = ds.where(ds.disturbance.isnull(), drop=True)
undisturbed = undisturbed.where(np.abs(undisturbed.digital_elevation_model - undisturbed.elev_lowestmode) < 50,\
     drop=True)
undisturbed = undisturbed.where(undisturbed.agbd < 500, drop=True)

In [ ]:
dists = [40, 100, 200, 400] #200

def fix_dict_keys(v):
    if isinstance(v, dict):
        return {str(k): val for k, val in v.items()}
    return v

for dist in dists:
    
    hp_df = pd.DataFrame([])

    for w in [0, 0.25, 0.5, 0.75]: 
        
        f = f'data/sp_stratified_samples/pairs_{dist}m_stratified_laz.parquet'
        subsample = f'data/sp_stratified_samples/sim_all_{dist}m.csv'

        subsample = gpd.read_file(subsample)

        subs_indices = list(pd.to_numeric(subsample['index'])) + list(pd.to_numeric(subsample['index']) + 2150)
        
        geo = gpd.read_parquet(f).iloc[subs_indices] 

        if len(geo)//2 > 50:

            indices = np.arange(0, len(geo)//2 + 1, 50)

            for idx in range(len(indices) - 1):
                
                geo_ = geo.iloc[list(range(indices[idx], indices[idx+1])) + list(range(len(geo)//2 + indices[idx], len(geo)//2 + indices[idx+1]))] 

                geo_['shot_num_1'] = list(geo_['shot_num'].iloc[:len(geo_)//2].astype('str')) + list(np.nan * np.ones(len(geo_)//2))
                geo_['shot_num_2'] = list(np.nan * np.ones(len(geo_)//2)) + list(geo_['shot_num'].iloc[len(geo_)//2:].astype('str'))

                shots_2 = geo_.iloc[len(geo_)//2:]
                shots_1 = geo_.iloc[:len(geo_)//2]

                

                pairs = pairing_algorithms.get_close_fs_pairs(undisturbed, shots_2, 
                                                                weight_geo=w,
                                                                max_distance=400, 
                                                                use_baseline=True,
                                                                use_slope=True,
                                                                use_embeddings = False,
                                                                disturbed = False,
                                                                n_jobs = 10)
                
                pairs['shot_num_1'] = shots_1.shot_num_1.values
                pairs['weight'] = w
                pairs['max_distance'] = dist

                s1_new = undisturbed.sel(shot_number=np.uint64(pairs.new_shot_num_1))

                pairs['geometry'] = gpd.points_from_xy(s1_new.longitude.values, s1_new.latitude.values)

                hp_df = pd.concat([hp_df, pairs])
                hp_df['s2_feats'] = hp_df['s2_feats'].astype(str)
                hp_df['all_pos_feats'] = hp_df['all_pos_feats'].astype(str)

                hp_df.to_parquet((f'data/fs_stratified_samples/rq_2_fs_pairs_{dist}m.parquet'))

        else:

            geo['shot_num_1'] = list(geo['shot_num'].iloc[:len(geo)//2].astype('str')) + list(np.nan * np.ones(len(geo)//2))
            geo['shot_num_2'] = list(np.nan * np.ones(len(geo)//2)) + list(geo['shot_num'].iloc[len(geo)//2:].astype('str'))

            shots_2 = geo.iloc[len(geo)//2:]
            shots_1 = geo.iloc[:len(geo)//2]

            pairs = pairing_algorithms.get_close_fs_pairs(undisturbed, shots_2, 
                                                        weight_geo=w,
                                                        max_distance=400, # Change to dist when running for within bin
                                                        use_baseline=True,
                                                        use_slope=True,
                                                        use_embeddings = False,
                                                        disturbed = False,
                                                        n_jobs = 10)
            
            pairs['shot_num_1'] = shots_1.shot_num_1.values
            pairs['weight'] = w
            pairs['max_distance'] = dist

            s1_new = undisturbed.sel(shot_number=np.uint64(pairs.new_shot_num_1))

            pairs['geometry'] = gpd.points_from_xy(s1_new.longitude.values, s1_new.latitude.values)

            hp_df = pd.concat([hp_df, pairs])
            hp_df['s2_feats'] = hp_df['s2_feats'].astype(str)
            hp_df['all_pos_feats'] = hp_df['all_pos_feats'].astype(str)

            hp_df.to_parquet(f'data/fs_stratified_samples/rq_2_fs_pairs_{dist}m.parquet')
